# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [57]:
import pandas as pd
PATH = '/content/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head(5)

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [58]:
# TODO (hint: .str.lower().str.contains('pop'))
cat[cat['product_name'].str.lower().str.contains('pop')]
print('There are',len(cat[cat['product_name'].str.lower().str.contains('pop')]),'rows that look like Pop-Tart products')
print(cat[cat['product_name'].str.lower().str.contains('pop')])

There are 4 rows that look like Pop-Tart products
             sku          product_name         category subcategory  \
0          PT-12  pop-tarts strawberry  Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart        Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast   Pop-Tarts   

   unit_cost           supplier package_size  
0       1.93  Johnson & Johnson       4-pack  
1       2.26           Roku Inc          4pk  
2       2.11           Sara Lee       6 pack  
3        NaN         Campbell's       4 pack  


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [59]:
# TODO
cat[cat['product_name'].str.lower().str.contains('strawberry')]

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


In [60]:
print('There are',cat.duplicated().sum(),'duplicated rows')

There are 0 duplicated rows


**Pop-Tart rows that are a different product:** Zero?

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [61]:
out = cat.drop_duplicates().copy()
CANON = {
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW',
}

def consolidate(df):
    df_copy = df.copy()
    df_copy['sku_original'] = df_copy['sku']
    df_copy['canonical_sku'] = df_copy['sku'].map(CANON).fillna(df_copy['sku'])
    df_copy['sku'] = df_copy['canonical_sku']
    return df_copy

out = consolidate(cat)
out.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size,sku_original,canonical_sku
0,POPTART-STRAW,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,PT-12,POPTART-STRAW
1,POPTART-STRAW,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,SKU#459812,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,POPTART-STRAW,POPTART-STRAW
3,POPTART-STRAW,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack,PT_STRAWBERRY,POPTART-STRAW
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack,WTR-24,WTR-24


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [62]:
# TODO
print(out['category'].unique())
out['category'] = out['category'].str.strip().str.title()
print(out['category'].unique())
out.head()

['Breakfast Foods' 'Breakfast' 'breakfast' 'Emergency Supplies'
 'Emergency' 'EMERGENCY' 'Alcoholic Beverages' 'ALCOHOL' 'ENTERTAINMENT'
 'entertainment' 'Electronics' 'electronics' 'Groceries' 'GROCERY'
 'Grocery' 'grocery' 'health' 'ELECTRONICS' 'snacks' 'Snacks' 'household'
 'HOUSEHOLD' 'Household' 'baby' 'Baby Products' 'Baby' 'emergency']
['Breakfast Foods' 'Breakfast' 'Emergency Supplies' 'Emergency'
 'Alcoholic Beverages' 'Alcohol' 'Entertainment' 'Electronics' 'Groceries'
 'Grocery' 'Health' 'Snacks' 'Household' 'Baby' 'Baby Products']


,sku,product_name,category,subcategory,unit_cost,supplier,package_size,sku_original,canonical_sku
0,POPTART-STRAW,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,PT-12,POPTART-STRAW
1,POPTART-STRAW,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,SKU#459812,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,POPTART-STRAW,POPTART-STRAW
3,POPTART-STRAW,pop-tarts strawberry,Breakfast,Pop-Tarts,NaN,Campbell's,4 pack,PT_STRAWBERRY,POPTART-STRAW
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack,WTR-24,WTR-24


In [63]:
before = out['category'].value_counts()
print(before, '\n', before.shape[0], 'distinct before\n')

out['category'] = out['category'].str.strip().str.title()
after = out['category'].value_counts()
print(after, '\n', after.shape[0], 'distinct after title-case\n')

CAT_MAP = {
    'Breakfast Foods':     'Breakfast',
    'Emergency Supplies':  'Emergency',
    'Groceries':           'Grocery',
    'Alcoholic Beverages': 'Alcohol',
    'Baby Products':       'Baby',
}
out['category'] = out['category'].replace(CAT_MAP)   # unmapped values stay as they are
print(out['category'].value_counts(), '\n', out['category'].nunique(), 'distinct final')

category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast Foods        1
Alcohol                1
Alcoholic Beverages    1
Groceries              1
Baby Products          1
Name: count, dtype: int64 
 15 distinct before

category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast Foods        1
Alcohol                1
Alcoholic Beverages    1
Groceries              1
Baby Products          1
Name: count, dtype: int64 
 15 distinct after title-case

category
Emergency        12
Grocery           6
Entertainment     5
Breakfast         4
Electronics       4
Baby             

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [64]:
cat['product_name'].unique()

array(['pop-tarts strawberry', 'Strawberry Pop-Tart',
       'Pop-Tarts Strawberry', 'BOTTLED WATER 24PK',
       'Bottled Water 24-Pack', 'Water - 24 Pack', 'BATTERIES AA 4PK',
       'AA Batteries 4-Pack', 'FLASHLIGHT LED', 'Light Beer (24-pack)',
       'Monopoly Board Game', 'Board Game - Monopoly', 'Playing Cards',
       'PLAYING CARDS', 'Roku Streaming Stick', 'Streaming Device',
       'Soup - Canned', 'Canned Soup', 'WHITE BREAD', 'PEANUT BUTTER',
       'Peanut Butter', 'Candles (Emergency)', 'Emergency Candles',
       'First-Aid Kit', 'First Aid Kit', 'PORTABLE CHARGER',
       'Portable Phone Charger', 'Variety Pack - Chips',
       'Chips Variety Pack', 'Paper Towels', 'TOILET PAPER 12PK',
       'Toilet Paper 12-Roll', 'Baby Diapers - Size 3', 'DIAPERS SZ 3',
       'Infant Formula', 'Baby Formula', 'Heavy Duty Tarp',
       'TARP - HEAVY DUTY'], dtype=object)

In [65]:
# TODO
catalog = pd.DataFrame({
    'product_name': ['pop-tarts strawberry', 'Strawberry Pop-Tart',
       'Pop-Tarts Strawberry', 'BOTTLED WATER 24PK',
       'Bottled Water 24-Pack', 'Water - 24 Pack', 'BATTERIES AA 4PK',
       'AA Batteries 4-Pack', 'FLASHLIGHT LED', 'Light Beer (24-pack)',
       'Monopoly Board Game', 'Board Game - Monopoly', 'Playing Cards',
       'PLAYING CARDS', 'Roku Streaming Stick', 'Streaming Device',
       'Soup - Canned', 'Canned Soup', 'WHITE BREAD', 'PEANUT BUTTER',
       'Peanut Butter', 'Candles (Emergency)', 'Emergency Candles',
       'First-Aid Kit', 'First Aid Kit', 'PORTABLE CHARGER',
       'Portable Phone Charger', 'Variety Pack - Chips',
       'Chips Variety Pack', 'Paper Towels', 'TOILET PAPER 12PK',
       'Toilet Paper 12-Roll', 'Baby Diapers - Size 3', 'DIAPERS SZ 3',
       'Infant Formula', 'Baby Formula', 'Heavy Duty Tarp',
       'TARP - HEAVY DUTY'],
})

# Use code to narrow it down: group by a normalized name and look for matches
catalog['normalized'] = (catalog['product_name'].str.lower()
                         .str.replace(r'[^a-z ]', '', regex=True)
                         .str.split().apply(lambda w: ' '.join(sorted(w))))

candidates = (catalog.groupby('normalized')['product_name']
              .agg(list).loc[lambda s: s.str.len() > 1])
candidates

,product_name
normalized,
board game monopoly,"[Monopoly Board Game, Board Game - Monopoly]"
butter peanut,"[PEANUT BUTTER, Peanut Butter]"
candles emergency,"[Candles (Emergency), Emergency Candles]"
canned soup,"[Soup - Canned, Canned Soup]"
cards playing,"[Playing Cards, PLAYING CARDS]"
chips pack variety,"[Variety Pack - Chips, Chips Variety Pack]"
duty heavy tarp,"[Heavy Duty Tarp, TARP - HEAVY DUTY]"
poptarts strawberry,"[pop-tarts strawberry, Pop-Tarts Strawberry]"


### Google Gemini Explanation of why some items are not in the candidates list:

The code identifies potential duplicates (fragmentation) by grouping together products that resolve to the exact same normalized word signature.

At the end of the script, there is a filter:
`loc[lambda s: s.str.len() > 1]`

This specifically keeps **only** those groups that contain **more than one** unique product name (i.e., actual candidates for merging). Any product name that normalizes to a unique signature with only a single associated product name is excluded from this list because it does not have an obvious duplicate under this normalization scheme.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [66]:
# TODO
assert out['canonical_sku'].isnull().sum() == 0
assert out.shape[0] == cat.shape[0]
before_rows = len(cat)
out = consolidate(cat)          # returns a copy with canonical_sku added

# 1. No rows added or removed
assert len(out) == before_rows, f"Row count changed: {before_rows} -> {len(out)}"

# 2. No nulls introduced in the canonical column
assert out['canonical_sku'].notna().all(), \
    f"{out['canonical_sku'].isna().sum()} null canonical_sku values"

print("OK:", len(out), "rows,", out['canonical_sku'].isna().sum(), "nulls")

# 3. Unmapped SKUs are untouched
unmapped = ~out['sku_original'].isin(CANON.keys())
assert (out.loc[unmapped, 'canonical_sku'] == out.loc[unmapped, 'sku_original']).all()

# 4. Only the intended rows changed
changed = out[out['canonical_sku'] != out['sku_original']]
print(len(changed), "rows relabeled:", changed['sku_original'].tolist())

# 5. Every mapping target actually exists in the data
assert set(CANON.values()) <= set(cat['sku']), "mapping points to a SKU that isn't in the catalog"

# 6. Idempotent: running it twice gives the same result
assert consolidate(out.assign(sku=out['canonical_sku']))['canonical_sku'].equals(out['canonical_sku'])

OK: 45 rows, 0 nulls
3 rows relabeled: ['PT-12', 'SKU#459812', 'PT_STRAWBERRY']


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

A) Bottled water, AA batteries and streaming devices are also fragmented. For bottled water and the AA batteries, they are 24 and 4 pack respectively. I would group all the water bottles together as 24 pack water and AA batteries as 4 pack AA batteries. There may be an issue if the prices are different between product, but if the task is to simply know the quantity of what is wanted, that grouping would be helpul.


---

B) An example would be what I was discussing in the previous question. There are three different options for a 24 pack of water. With the strawberry poptart, they are the exact samne product so they can be grouped the same. With the water, they are supplied by different companies and have different prices. If the task is to understand which specific brand the consumers desire, it is crucial that the grouping is not made based simply on being a 24 pack of water. Additionally, because the data set includes information on suppliers, it may be more viable to source from one supplier than another given the weather emergency. I would probably ask a senior data scientist or a senior person in operations for clarification on what the task is.